In [1]:
!pip -q install torch torchvision

import torch
import torch.nn as nn
import torchvision
import torchvision.models as models

import numpy as np
import os
import json
import math

print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)

PyTorch version: 2.11.0+cpu
Torchvision version: 0.26.0+cpu


In [2]:
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights

weights = MobileNet_V2_Weights.DEFAULT

model = mobilenet_v2(weights=weights)
model.eval()

print(model)

Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 99.0MB/s]


MobileNetV2(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU6(inplace=True)
        )
        (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (2): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(96, eps=

In [3]:
print("Convolution layers:\n")

for name, module in model.named_modules():
    if isinstance(module, nn.Conv2d):
        print(
            f"{name:30s} "
            f"in={module.in_channels:4d} "
            f"out={module.out_channels:4d} "
            f"kernel={module.kernel_size} "
            f"stride={module.stride}"
        )

Convolution layers:

features.0.0                   in=   3 out=  32 kernel=(3, 3) stride=(2, 2)
features.1.conv.0.0            in=  32 out=  32 kernel=(3, 3) stride=(1, 1)
features.1.conv.1              in=  32 out=  16 kernel=(1, 1) stride=(1, 1)
features.2.conv.0.0            in=  16 out=  96 kernel=(1, 1) stride=(1, 1)
features.2.conv.1.0            in=  96 out=  96 kernel=(3, 3) stride=(2, 2)
features.2.conv.2              in=  96 out=  24 kernel=(1, 1) stride=(1, 1)
features.3.conv.0.0            in=  24 out= 144 kernel=(1, 1) stride=(1, 1)
features.3.conv.1.0            in= 144 out= 144 kernel=(3, 3) stride=(1, 1)
features.3.conv.2              in= 144 out=  24 kernel=(1, 1) stride=(1, 1)
features.4.conv.0.0            in=  24 out= 144 kernel=(1, 1) stride=(1, 1)
features.4.conv.1.0            in= 144 out= 144 kernel=(3, 3) stride=(2, 2)
features.4.conv.2              in= 144 out=  32 kernel=(1, 1) stride=(1, 1)
features.5.conv.0.0            in=  32 out= 192 kernel=(1, 1) strid

In [4]:
selected_name = None
selected_layer = None

for name, module in model.named_modules():
    if isinstance(module, nn.Conv2d):
        if module.kernel_size == (1, 1):
            selected_name = name
            selected_layer = module
            break

print("Selected layer:", selected_name)
print(selected_layer)

Selected layer: features.1.conv.1
Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)


In [5]:
# Random input image for initial hardware verification
# Later you can replace this with an actual dataset image.

input_image = torch.randn(1, 3, 224, 224)

captured_input = None

def hook_fn(module, inputs, output):
    global captured_input
    captured_input = inputs[0].detach().cpu()

hook = selected_layer.register_forward_hook(hook_fn)

with torch.no_grad():
    _ = model(input_image)

hook.remove()

print("Input feature map shape:", captured_input.shape)

Input feature map shape: torch.Size([1, 32, 112, 112])


In [7]:
weights_fp32 = selected_layer.weight.detach().cpu()

print("Weight shape:", weights_fp32.shape)
print("Minimum:", weights_fp32.min().item())
print("Maximum:", weights_fp32.max().item())
W = weights_fp32.squeeze(-1).squeeze(-1)

print("GEMM weight matrix shape:", W.shape)

Weight shape: torch.Size([16, 32, 1, 1])
Minimum: -1.8763929605484009
Maximum: 2.142815113067627
GEMM weight matrix shape: torch.Size([16, 32])


In [8]:
X = captured_input[0]

Cin, H, W_img = X.shape

# Convert feature map into matrix
X_gemm = X.reshape(Cin, H * W_img)

# Weight matrix
W_gemm = W

print("X GEMM shape:", X_gemm.shape)
print("W GEMM shape:", W_gemm.shape)


X GEMM shape: torch.Size([32, 12544])
W GEMM shape: torch.Size([16, 32])


In [10]:
# Get the actual output of the selected convolution
with torch.no_grad():
    Y_conv = selected_layer(captured_input)

print("Actual convolution output shape:", Y_conv.shape)

Actual convolution output shape: torch.Size([1, 16, 112, 112])


In [11]:
# captured_input shape:
# [1, Cin, H, W]

X = captured_input[0]

Cin, H, W_img = X.shape

# Convert input feature map to GEMM matrix
X_gemm = X.reshape(Cin, H * W_img)

# Weight matrix for 1x1 convolution
W_gemm = selected_layer.weight.detach().cpu().squeeze(-1).squeeze(-1)

# GEMM
Y_gemm = W_gemm @ X_gemm

# Add bias
if selected_layer.bias is not None:
    Y_gemm = Y_gemm + selected_layer.bias.detach().cpu().reshape(-1, 1)

# Reshape GEMM result back to convolution output format
Y_gemm_4d = Y_gemm.reshape(
    1,
    selected_layer.out_channels,
    H,
    W_img
)

print("GEMM output shape:", Y_gemm_4d.shape)

GEMM output shape: torch.Size([1, 16, 112, 112])


In [12]:
difference = torch.abs(Y_conv - Y_gemm_4d)

print("Maximum difference:", difference.max().item())
print("Mean difference:", difference.mean().item())

Maximum difference: 0.0
Mean difference: 0.0


In [13]:
print("Layer:", selected_name)
print("Kernel:", selected_layer.kernel_size)
print("Stride:", selected_layer.stride)
print("Padding:", selected_layer.padding)
print("Input shape:", captured_input.shape)
print("Weight shape:", selected_layer.weight.shape)

Layer: features.1.conv.1
Kernel: (1, 1)
Stride: (1, 1)
Padding: (0, 0)
Input shape: torch.Size([1, 32, 112, 112])
Weight shape: torch.Size([16, 32, 1, 1])


In [16]:
def quantize_int8(x):
    x = x.detach().cpu()

    max_val = torch.max(torch.abs(x))

    if max_val == 0:
        scale = 1.0
    else:
        scale = max_val.item() / 127.0

    q = torch.round(x / scale)
    q = torch.clamp(q, -128, 127)

    return q.to(torch.int8), scale
X_int8, X_scale = quantize_int8(X_gemm)

# Quantize weights
W_int8, W_scale = quantize_int8(W_gemm)

print("Input scale :", X_scale)
print("Weight scale:", W_scale)

print("Input INT8 range:",
      X_int8.min().item(),
      X_int8.max().item())

print("Weight INT8 range:",
      W_int8.min().item(),
      W_int8.max().item())

Input scale : 0.047244094488188976
Weight scale: 0.016872559945414384
Input INT8 range: 0 127
Weight INT8 range: -111 127


In [17]:
X_int32 = X_int8.to(torch.int32)
W_int32 = W_int8.to(torch.int32)

Y_int32 = W_int32 @ X_int32

print("INT32 output shape:", Y_int32.shape)

print("Accumulator range:")
print("min =", Y_int32.min().item())
print("max =", Y_int32.max().item())

INT32 output shape: torch.Size([16, 12544])
Accumulator range:
min = -29236
max = 40400


In [19]:
# ============================================
# CELL 8: Convert 1x1 convolution to GEMM
# ============================================

# Input feature map
X = captured_input[0]

Cin, H, W_img = X.shape

# Convert feature map to matrix
X_gemm = X.reshape(Cin, H * W_img)

# Convert 1x1 convolution weights to matrix
W_gemm = selected_layer.weight.detach().cpu().squeeze(-1).squeeze(-1)

# GEMM
Y_gemm = W_gemm @ X_gemm

# Add bias if present
if selected_layer.bias is not None:
    Y_gemm = Y_gemm + selected_layer.bias.detach().cpu().reshape(-1, 1)

# Reshape back to CNN format
Y_gemm_4d = Y_gemm.reshape(
    1,
    selected_layer.out_channels,
    H,
    W_img
)

print("Input GEMM shape :", X_gemm.shape)
print("Weight GEMM shape:", W_gemm.shape)
print("Output GEMM shape:", Y_gemm_4d.shape)

Input GEMM shape : torch.Size([32, 12544])
Weight GEMM shape: torch.Size([16, 32])
Output GEMM shape: torch.Size([1, 16, 112, 112])


In [20]:
# ============================================
# CELL 9: Verify GEMM against actual convolution
# ============================================

with torch.no_grad():
    Y_conv = selected_layer(captured_input)

difference = torch.abs(Y_conv - Y_gemm_4d)

print("Maximum difference:", difference.max().item())
print("Mean difference   :", difference.mean().item())

Maximum difference: 0.0
Mean difference   : 0.0


In [23]:
# ============================================
# CELL 11: INT8 GEMM and Error Analysis
# ============================================

# Convert INT8 values to INT32 before multiplication
X_int32 = X_int8.to(torch.int32)
W_int32 = W_int8.to(torch.int32)

# Hardware-style matrix multiplication
# INT8 × INT8 → INT32 accumulator
Y_int32 = W_int32 @ X_int32

print("INT32 output shape:", Y_int32.shape)

print("\nAccumulator range:")
print("Minimum:", Y_int32.min().item())
print("Maximum:", Y_int32.max().item())

# Dequantize INT32 result back to approximate FP32
Y_dequant = Y_int32.float() * X_scale * W_scale

# Compare against the original FP32 GEMM result
Y_fp32_reference = Y_gemm

error = torch.abs(Y_fp32_reference - Y_dequant)

print("\nQuantization error:")
print("Maximum absolute error:", error.max().item())
print("Mean absolute error   :", error.mean().item())

INT32 output shape: torch.Size([16, 12544])

Accumulator range:
Minimum: -29236
Maximum: 40400

Quantization error:
Maximum absolute error: 0.34957385063171387
Mean absolute error   : 0.07453995198011398


In [21]:
# ============================================
# CELL 10: INT8 Quantization
# ============================================

def quantize_int8(x):
    x = x.detach().cpu()

    max_val = torch.max(torch.abs(x))

    if max_val == 0:
        scale = 1.0
    else:
        scale = max_val.item() / 127.0

    q = torch.round(x / scale)
    q = torch.clamp(q, -128, 127)

    return q.to(torch.int8), scale


X_int8, X_scale = quantize_int8(X_gemm)
W_int8, W_scale = quantize_int8(W_gemm)

print("Input scale :", X_scale)
print("Weight scale:", W_scale)

print("Input INT8 range:",
      X_int8.min().item(),
      X_int8.max().item())

print("Weight INT8 range:",
      W_int8.min().item(),
      W_int8.max().item())

Input scale : 0.047244094488188976
Weight scale: 0.016872559945414384
Input INT8 range: 0 127
Weight INT8 range: -111 127


In [24]:
# ============================================
# CELL 12: Generate .mem files for Vivado
# ============================================

import os

# Create output folder
os.makedirs("vivado_mem", exist_ok=True)


# --------------------------------------------
# INT8 → 8-bit hexadecimal
# --------------------------------------------
def int8_to_hex(value):
    value = int(value)

    # Convert signed INT8 to unsigned 8-bit representation
    if value < 0:
        value += 256

    return f"{value:02X}"


# --------------------------------------------
# INT32 → 32-bit hexadecimal
# --------------------------------------------
def int32_to_hex(value):
    value = int(value)

    # Convert signed INT32 to 32-bit two's complement
    if value < 0:
        value += (1 << 32)

    return f"{value:08X}"


# ============================================
# 1. INPUT MEMORY FILE
# ============================================

input_path = "vivado_mem/input.mem"

with open(input_path, "w") as f:

    for value in X_int8.flatten().numpy():
        f.write(int8_to_hex(value) + "\n")


# ============================================
# 2. WEIGHT MEMORY FILE
# ============================================

weights_path = "vivado_mem/weights.mem"

with open(weights_path, "w") as f:

    for value in W_int8.flatten().numpy():
        f.write(int8_to_hex(value) + "\n")


# ============================================
# 3. GOLDEN OUTPUT MEMORY FILE
# ============================================

golden_path = "vivado_mem/golden_output.mem"

with open(golden_path, "w") as f:

    for value in Y_int32.flatten().numpy():
        f.write(int32_to_hex(value) + "\n")


print("======================================")
print("Vivado memory files generated")
print("======================================")

print("Input file   :", input_path)
print("Weight file  :", weights_path)
print("Golden output:", golden_path)

Vivado memory files generated
Input file   : vivado_mem/input.mem
Weight file  : vivado_mem/weights.mem
Golden output: vivado_mem/golden_output.mem


In [25]:
# ============================================
# CELL 13: Verify generated .mem files
# ============================================

def show_file_info(path, name, num_lines=10):

    with open(path, "r") as f:
        lines = f.readlines()

    print("\n" + "=" * 50)
    print(name)
    print("=" * 50)

    print("Number of entries:", len(lines))
    print("First", num_lines, "entries:")

    for line in lines[:num_lines]:
        print(line.strip())


show_file_info(
    "vivado_mem/input.mem",
    "INPUT.MEM"
)

show_file_info(
    "vivado_mem/weights.mem",
    "WEIGHTS.MEM"
)

show_file_info(
    "vivado_mem/golden_output.mem",
    "GOLDEN_OUTPUT.MEM"
)


INPUT.MEM
Number of entries: 401408
First 10 entries:
00
01
00
00
00
00
00
0F
00
00

WEIGHTS.MEM
Number of entries: 512
First 10 entries:
04
0D
02
1A
FF
F9
00
EE
FF
00

GOLDEN_OUTPUT.MEM
Number of entries: 200704
First 10 entries:
FFFFF3DF
FFFFFB9B
000009D4
FFFFF4A3
00000F23
FFFFDD6C
FFFFF644
000016AD
FFFFFFAF
00001A39


In [26]:
# ============================================
# CELL 14: Check matrix dimensions
# ============================================

print("X_int8 shape      :", X_int8.shape)
print("W_int8 shape      :", W_int8.shape)
print("Y_int32 shape     :", Y_int32.shape)

print()

print("Expected input values  :", X_int8.numel())
print("Expected weight values :", W_int8.numel())
print("Expected output values :", Y_int32.numel())

print()

print("Actual input.mem lines  :",
      sum(1 for _ in open("vivado_mem/input.mem")))

print("Actual weights.mem lines:",
      sum(1 for _ in open("vivado_mem/weights.mem")))

print("Actual golden lines     :",
      sum(1 for _ in open("vivado_mem/golden_output.mem")))

X_int8 shape      : torch.Size([32, 12544])
W_int8 shape      : torch.Size([16, 32])
Y_int32 shape     : torch.Size([16, 12544])

Expected input values  : 401408
Expected weight values : 512
Expected output values : 200704

Actual input.mem lines  : 401408
Actual weights.mem lines: 512
Actual golden lines     : 200704


In [27]:
# ============================================
# CELL 15: Save quantization parameters
# ============================================

import json

quant_params = {
    "layer": selected_name,

    "input_scale": X_scale,
    "weight_scale": W_scale,

    "input_shape": list(X_int8.shape),
    "weight_shape": list(W_int8.shape),
    "output_shape": list(Y_int32.shape),

    "input_dtype": "INT8",
    "weight_dtype": "INT8",
    "accumulator_dtype": "INT32"
}

with open("vivado_mem/quantization.json", "w") as f:
    json.dump(quant_params, f, indent=4)

print(json.dumps(quant_params, indent=4))

{
    "layer": "features.1.conv.1",
    "input_scale": 0.047244094488188976,
    "weight_scale": 0.016872559945414384,
    "input_shape": [
        32,
        12544
    ],
    "weight_shape": [
        16,
        32
    ],
    "output_shape": [
        16,
        12544
    ],
    "input_dtype": "INT8",
    "weight_dtype": "INT8",
    "accumulator_dtype": "INT32"
}


In [28]:
# ============================================
# CELL 16: Download Vivado files
# ============================================

from google.colab import files
import shutil

shutil.make_archive(
    "vivado_mem_files",
    "zip",
    "vivado_mem"
)

files.download("vivado_mem_files.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>